# Module 5: one-day-ahead regression study

Run the cells in order using the complete source folder and the regression/visualization extras. Each stage writes reviewable files. The first real-data download requires internet; the cached dataset supports later offline runs. No earlier notebook state is needed.

The script route calls the same functions. Named text and CSV/HTML tables carry the evidence alongside optional figures. Notebook-interface accessibility remains a separate verification task. Save, restart the kernel, and run all before sharing.


In [ ]:
from pathlib import Path
import json
import sys
import tempfile
import numpy as np
import pandas as pd

# Locate shared source without requiring a particular OS path or copied code.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ml_ai_it"]
ROOT = next((p for p in candidates if (p / "src/practical_ml_ai").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete ml_ai_it source folder.")
sys.path.insert(0, str(ROOT / "src"))
from practical_ml_ai.regression import (
    download_bike_data, prepare_study, train_study, validate_study,
    evaluate_study, infer_study, report_study, load_split, leakage_demo,
)
print("Shared source:", ROOT)


## 1. Choose the data explicitly

The core uses the actual UCI Bike Sharing daily data (CC BY 4.0). The downloader retains attribution, the original archive/CSV, and reviewed checksums. It reuses a valid cache. Change DATASET to synthetic only for an explicitly labeled offline practice run; no automatic substitution is made.


In [ ]:
DATASET = "uci"  # Alternatives: "uci" or "synthetic"; explain your choice in the report.
MAKE_PLOTS = True  # False preserves text/tables without the optional Matplotlib figures.
if DATASET == "uci":
    input_path = download_bike_data(ROOT / "runs/datasets/bike-sharing")
    holdout_days = 110
    dataset_label = "UCI Bike Sharing daily counts, CC BY 4.0"
    print("Dataset provenance:")
    print((input_path.parent / "provenance.json").read_text(encoding="utf-8"))
elif DATASET == "synthetic":
    input_path = ROOT / "data/samples/forecast-demand/daily.csv"
    holdout_days = 14
    dataset_label = "Original synthetic smoke fixture"
else:
    raise ValueError("Choose uci or synthetic explicitly.")
print("Dataset:", dataset_label)


## 2. Check arithmetic before fitting

This hand example is separate from fitted model results. Predict the residual signs and MAE first. Positive residual means the prediction was too low. A small mean residual can hide cancellation.


In [ ]:
# Two illustrative observations, not UCI model parameters.
actual = np.array([110.0, 125.0])
predicted = np.array([100.0, 140.0])
hand_residuals = actual - predicted
hand_mae = np.abs(hand_residuals).mean()
print("Residuals (actual minus predicted):", hand_residuals)
print("Mean residual:", hand_residuals.mean())
print("MAE:", hand_mae)


## 3. Prepare a fresh study

Forecast tomorrow after today's count is finalized. The prepared features use counts from target minus one and minus seven days. The first seven days supply history. Target dates remain chronological.

The notebook creates a new generated directory each run because frozen stages should not be overwritten. Repeated runs on public historical data are demonstrations, not independent new tests.


In [ ]:
(ROOT / "runs").mkdir(exist_ok=True)
study_parent = Path(tempfile.mkdtemp(prefix="module05-notebook-", dir=ROOT / "runs"))
run_dir = study_parent / "study"  # This child does not exist yet; preparation creates it.
manifest = prepare_study(input_path, run_dir, holdout_days=holdout_days)
print("Save your generated evidence from:", run_dir)
for name, details in manifest["splits"].items():
    print(name, "rows:", details["rows"], "target dates:", details["first_target"], "through", details["last_target"])
print("Source SHA-256:", manifest["source_sha256"])


## 4. Fit training-only pipelines and baselines

The training command opens only training data. X contains features and y contains the target. The actual shared fit uses a median-imputer/LinearRegression pipeline; fill values and coefficients are learned on training only. Baselines use the training mean or already observed lags. Inspect the artifact rather than assuming every method is a learned model.


In [ ]:
training_preview = load_split(run_dir, "train")
X = training_preview[["prior_week_requests"]]  # Double brackets retain a 2-D feature matrix.
y = training_preview["target_requests"]
print("X shape:", X.shape, "y shape:", y.shape)

trained = train_study(run_dir)
print("Library versions:", trained["versions"])
print("Weekly linear model parameters:")
print(json.dumps(trained["models"]["linear_week"], indent=2))
print("All predefined candidates:", list(trained["models"]))


## 5. Validate and freeze the choice

Validation compares predefined candidates; a baseline may win. Record this choice before opening final scores. The core retains training-only fitted parameters after selection and does not silently refit on training plus validation.


In [ ]:
selection = validate_study(run_dir)
validation_scores = pd.read_csv(run_dir / "validation_scores.csv")
print(validation_scores.to_string(index=False))
print("Frozen selection:", selection["selected"])
print("Selection rule:", selection["rule"])


## 6. Deliberately invalid leakage demonstration

The following example supplies tomorrow's actual count as its own feature. Its nearly perfect error is unusable because that input would not exist at forecast origin. This demonstration reads training/validation only and is never added to the candidate or selected artifacts.


In [ ]:
invalid_example = leakage_demo(
    load_split(run_dir, "train"), load_split(run_dir, "validation")
)
print("INVALID validation MAE:", invalid_example["invalid_validation_mae"])
print(invalid_example["warning"])
print("The valid candidate names remain:", list(trained["models"]))


## 7. Evaluate once and inspect residuals

The final stage applies frozen parameters. It reports the chosen method and predefined baselines; it does not select another winner. Later one-day-ahead origins can use earlier holdout days once those days have actually been observed. This is not a multi-day forecast issued entirely before the holdout.

Read the full selected-prediction CSV if you cannot use the figures. Identify a large error and investigate its context without making an unsupported causal claim.


In [ ]:
result = evaluate_study(run_dir)
report_study(run_dir, plots=MAKE_PLOTS, dataset_label=dataset_label)
selected_predictions = pd.read_csv(run_dir / "selected_predictions.csv")
print((run_dir / "summary.txt").read_text(encoding="utf-8"))
print("Final scores (not a model-selection table):")
print(pd.DataFrame(result["scores"]).to_string(index=False))
print("First selected predictions:")
print(selected_predictions.head().to_string(index=False))
print("Report with the complete table:", run_dir / "index.html")


## 8. Inference without a supplied actual target

The future-feature row uses only already observed counts. It has a forecast date but no target label, so this step produces a prediction without an error score. The script writes forecast.csv. Real-valued output is retained; rounding/clipping would need a declared application policy.


In [ ]:
future_features = pd.read_csv(run_dir / "future_features.csv")
forecast = infer_study(run_dir)
print("Features available at the future origin:")
print(future_features.to_string(index=False))
print("Unscored future prediction:")
print(forecast.to_string(index=False))


## Optional depth: expanding windows within training

The training stage also generated three expanding-window CV results, each fitting a fresh pipeline on earlier training rows. Inspect the increasing fit windows and later check dates. These folds do not use validation or final-test rows and are not an excuse to tune against the final report.


In [ ]:
folds = pd.read_csv(run_dir / "training_cv.csv")
print(folds.to_string(index=False))
print("Each last fit target precedes its first check target:",
      (folds["last_fit_target"] < folds["first_check_target"]).all())


## Save and explain

Complete the module practice and stakeholder memo. Retain the source provenance, split manifest, candidate/selection artifacts, commands, versions, and named results. Save this notebook, restart the kernel, and run all cells. A new output directory supports replay, but the same public final data are not fresh evaluation evidence. Follow your instructor's AI-use and submission rules.
